In [1]:
# ===============================================================
#              ELSS Anomaly Detection Pipeline
# ===============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

import matplotlib.pyplot as plt
import seaborn as sns

# ----------------- CONFIG -----------------
INPUT_CSV = "engineered_features_elss.csv"
GROUP_COL = "Scheme Code"
DATE_COL  = "Date"
NAV_COL   = "NAV"

OUT_DIR = "anomaly_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42
CONTAM = 0.01       # 1% anomalies expected
# ---------------------------------------------------------------

# 1) Load data
df = pd.read_csv(INPUT_CSV)
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df = df.sort_values([GROUP_COL, DATE_COL]).reset_index(drop=True)
print("Loaded:", df.shape)

# 2) Choose anomaly features
# NAV-based sudden changes are the most important
anomaly_features = [
    NAV_COL,
    "nav_diff_1",
    "nav_roll_mean_7",
    "nav_roll_std_7",
    "nav_roll_mean_30",
    "nav_expanding_std",
    "target_next_return"
]

# Keep only those that exist
anomaly_features = [c for c in anomaly_features if c in df.columns]

print("Using anomaly features:", anomaly_features)

# 3) Prepare output container
df["iso_score"] = np.nan
df["iso_flag"]  = 0

df["lof_score"] = np.nan
df["lof_flag"]  = 0

# 4) Detect anomalies PER SCHEME
unique_schemes = df[GROUP_COL].unique()

for sc in unique_schemes:
    g = df[df[GROUP_COL] == sc].copy()
    
    if len(g) < 20:
        continue  # too little data
    
    X = g[anomaly_features].fillna(method="ffill").fillna(method="bfill")
    
    # -------------------------------------------------------
    # 4A) Isolation Forest (recommended)
    # -------------------------------------------------------
    iso = IsolationForest(
        contamination=CONTAM,
        random_state=RANDOM_STATE,
        n_estimators=200
    )
    
    iso.fit(X)
    scores = iso.decision_function(X)
    preds  = iso.predict(X)  # -1 = anomaly, 1 = normal
    
    df.loc[g.index, "iso_score"] = scores
    df.loc[g.index, "iso_flag"]  = (preds == -1).astype(int)
    
    # -------------------------------------------------------
    # 4B) Local Outlier Factor (LOF)
    # -------------------------------------------------------
    lof = LocalOutlierFactor(
        contamination=CONTAM,
        n_neighbors=20,
        novelty=False
    )
    
    lof_preds = lof.fit_predict(X)
    lof_scores = lof.negative_outlier_factor_
    
    df.loc[g.index, "lof_score"] = lof_scores
    df.loc[g.index, "lof_flag"]  = (lof_preds == -1).astype(int)

print("Anomaly detection completed.")

# 5) Save results
df.to_csv(os.path.join(OUT_DIR, "nav_anomaly_detection.csv"), index=False)
print("Saved:", os.path.join(OUT_DIR, "nav_anomaly_detection.csv"))

# 6) Summary stats
iso_total = df["iso_flag"].sum()
lof_total = df["lof_flag"].sum()

print("\nIsolation Forest anomalies:", iso_total)
print("LOF anomalies:", lof_total)

# 7) Show top anomalies (sorted by severity)
top_iso = df[df["iso_flag"] == 1].sort_values("iso_score").head(20)
top_lof = df[df["lof_flag"] == 1].sort_values("lof_score").head(20)

top_iso.to_csv(os.path.join(OUT_DIR, "top_iso_anomalies.csv"), index=False)
top_lof.to_csv(os.path.join(OUT_DIR, "top_lof_anomalies.csv"), index=False)

print("Top anomaly lists saved.")

# 8) Optional plot for one scheme
example_scheme = df[GROUP_COL].iloc[0]

plot_df = df[df[GROUP_COL] == example_scheme].copy()

plt.figure(figsize=(12,5))
sns.lineplot(data=plot_df, x=DATE_COL, y=NAV_COL, label="NAV")
sns.scatterplot(data=plot_df[plot_df["iso_flag"]==1],
                x=DATE_COL, y=NAV_COL, color="red", label="Anomaly")
plt.title(f"Example NAV anomalies — Scheme {example_scheme}")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f"nav_anomalies_scheme_{example_scheme}.png"))
plt.close()

print("Example plot saved.")


Loaded: (319086, 55)
Using anomaly features: ['NAV', 'nav_diff_1', 'nav_roll_mean_7', 'nav_roll_std_7', 'nav_roll_mean_30', 'nav_expanding_std', 'target_next_return']
Anomaly detection completed.
Saved: anomaly_outputs\nav_anomaly_detection.csv

Isolation Forest anomalies: 3287
LOF anomalies: 3287
Top anomaly lists saved.
Example plot saved.
